# 09b · 注册模型

## 这一章在整个项目里的位置

```
00–08 ✅ → [09 上线] → 10 对外 API
              │
              ├─ 09a 定版和考试 ✅
              ├─ 09b 注册模型：登记进注册表，过门禁，挂上"冠军"   ← 你在这里
              └─ 09c 全蛋白组打分
```

## 背景：为什么需要这一章

09a 结束时，两个最终模型躺在 MLflow 里，各有一个长长的地址（`models:/m-...`）。接下来要用它们给全蛋白组打分（09c），以后还会有 10 号的 API、还会有下一版模型。问题来了：
- 打分的程序该去哪里找模型？总不能把那个长地址抄进代码里
- 下个月训练出新版本，怎么换上去？换上去发现有问题，怎么换回来？
- 半年后有人问"网上这个分数是哪个模型算的、那个模型考试成绩多少"，怎么查？

**模型注册表**（model registry）就是回答这三个问题的。这一章做的事很短：**过门禁，登记，挂标签**。

## 你在哪里操作

```
你：在这个 notebook 里运行一格（几分钟）
看结果：Databricks 网页的 Catalog → qafi_dev → ml → Models
```

## 这一章做的 4 件事

| # | 做什么 | 比喻 |
|---|---|---|
| 1 | 注册表是什么 | 实验室的样品库：编号、版本、标签 |
| 2 | 门禁 | 没过质检的批次不入库 |
| 3 | 登记，挂上"冠军" | 入库，贴上"当前在用"的标签 |
| 4 | 按名字取回 | 用的人只认标签，不认批号 |

**花费：0**。**时间**：约 5 分钟。

---

## 准备

运行下面这一格。**不用看代码。**

In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "1"   # 这一章也会载入 PyTorch 和 XGBoost 的模型（见 07f）
import subprocess, warnings
from pathlib import Path
warnings.filterwarnings("ignore", message="IProgress not found")
warnings.filterwarnings("ignore", category=FutureWarning)
import pandas as pd
import torch
from databricks.sdk import WorkspaceClient
from qafi_platform import registry, tracking
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(ROOT)
def run(cmd, check=True):
    print(f"$ {cmd}")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(r.stdout + r.stderr)
    if check and r.returncode != 0:
        raise RuntimeError(f"failed: {cmd}")
w = WorkspaceClient(profile="qafi")
EXPERIMENT = f"/Users/{w.current_user.me().user_name}/qafi-variant-effect"
print("Databricks:", w.config.host)

---

## 第 1 步 · 注册表是什么

### 在干什么
弄清楚注册表里的三个词：名字、版本、标签。

### 背景
比喻：**实验室的样品库**。冰箱里一管细胞，要记清楚：
- **名字**：哪个细胞系
- **批号**：第几次冻存的（每冻一次一个新批号，旧的不会被覆盖）
- **标签**：贴在其中一管上的"当前实验用这管"。换一管用，就把标签撕下来贴到另一管上

模型注册表一模一样：

| 注册表的词 | 我们的例子 | 意思 |
|---|---|---|
| **名字** | `qafi_dev.ml.stability_head` | 一个模型的身份，和表放在同一个目录里（目录.楼层.名字） |
| **版本**（version） | 1、2、3…… | 每登记一次自动加一，永不覆盖 |
| **别名 / 标签**（alias） | `champion`（冠军） | 一个可以挪动的标签，同一时间只贴在一个版本上 |

**用模型的人只认标签**：09c 打分时写的是"给我 `stability_head@champion`"，不是"给我第 3 版"。以后换新模型，只要把 `champion` 挪到新版本上，打分程序一行不改。新版本出了问题，把标签挪回去，就是**回滚**。

### 用了什么技术，为什么
- **Unity Catalog 的模型注册表**：模型和表放在同一个目录体系里（`qafi_dev.ml.…`），所以权限、血缘、审计都和表一样管。在网页上点开一个模型，能一路追到：哪次 MLflow 运行训练的 → 用的哪一版特征表 → 特征表从哪些原始数据来
- 我们登记两个模型：
  - `qafi_dev.ml.stability_head`：稳定性网络（PyTorch）
  - `qafi_dev.ml.function_trees`：树模型（XGBoost），功能集成的成员之一

### 公司里什么情况怎么做
- 生产环境**只从注册表拿模型**，从不拿某个文件路径或某次运行。这是 MLOps 最基本的一条
- 通常有两个标签：`champion`（线上在用的）和 `challenger`（准备替换它、正在对比的）

### 面试怎么说
> "Released models live in the Unity Catalog model registry: a three-level name next to the tables, immutable numbered versions, and movable aliases. Consumers load name@champion, so a release or a rollback is moving an alias, not changing code."

---

## 第 2 步 · 门禁

### 在干什么
读 09a 那次考试的结果，三条规则全过才往下走。

### 背景
比喻：**没过质检的批次不入库**。09a 第 5 步把三条规则写进了考试记录：

| 规则 | 意思 |
|---|---|
| 稳定性分在稳定性实验上赢过零样本 | 训练出来的网络必须比不训练好 |
| 稳定性分在稳定性实验上赢过树模型 | 复杂的网络必须比简单的基线好 |
| 功能分在功能实验上赢过零样本 | 集成必须比不训练好 |

这一步只是**读结果**，不重新考。考试记录里还写明了考的是哪两个模型，下一步只登记这两个：**登记的就是考过的**，中间不可能被换掉。

### 执行

In [ ]:
PASSED, RULES, TESTED, TEST_RUN = registry.gate_result(EXPERIMENT)
display(RULES)
print("考过的模型:", TESTED)
assert PASSED, "门禁没过：不登记，不上线。看上面哪条规则没过，再决定下一步"
print("三条规则全部通过")

### 结果怎么看
- 表里三条规则，`passed` 一列都是 `True`，最后打印"三条规则全部通过"
- **如果有一条是 `False`**，这一格会在 `assert` 那里停下，后面的格子不该再跑。这时不是"改改模型再考"，而是：记下哪条没过、为什么，决定是否不上线，或者只上线过了的那部分（比如只发布稳定性分）

### 公司里什么情况怎么做
- 门禁要**自动执行**，不靠人记得去看。我们这里是一句 `assert`；在公司的流水线里，它是部署流程中的一步，不过就自动停下
- 门禁规则要在**看到考试结果之前**定好，09a 就是这么做的

### 面试怎么说
> "Registration reads the test evaluation and stops unless all release rules passed. The test run also records which models it evaluated, so only those can be registered."

---

## 第 3 步 · 登记，挂上"冠军"

### 在干什么
把考过的两个模型登记成新版本，贴上 `champion`。

### 背景
每个版本上还挂几个**标签**（tag），记录它的来历：哪次考试、门禁结果、代码版本。半年后有人问"这个模型凭什么上线"，点开版本就能看到。

### 执行

In [ ]:
VERSIONS = {}
for key, uri in TESTED.items():
    name = registry.full_name(key)
    VERSIONS[key] = registry.register(uri, name, description="Released in chapter 09: passed every release rule on the test proteins",
                                      tags={"test_run": TEST_RUN, "gate": "passed", "code_version": tracking.code_version()})
    registry.promote(name, VERSIONS[key], alias="champion")
    print(f"{name}: 第 {VERSIONS[key]} 版，已挂上 champion")
pd.concat({registry.full_name(k): registry.versions(registry.full_name(k)) for k in TESTED}).droplevel(1)

### 结果怎么看
- 两行打印：两个模型各自登记成了第几版、挂上了 `champion`。第一次运行是第 1 版
- 下面的表是两个模型的全部版本：`aliases` 一列里有 `champion`；`run_id` 指回训练它的那次 MLflow 运行；`tag test_run` 指回那次考试

### 去网页上看
Databricks → **Catalog** → `qafi_dev` → `ml` → **Models**（在表的旁边）：
- 点开 `stability_head`：能看到版本列表、`champion` 标签
- 点开第 1 版：**Source run** 一路链到 MLflow 的那次运行；**Lineage** 里能看到它和 `gold.fct_variant_features` 的关系

### 公司里什么情况怎么做
- **回滚**就是一句话：`registry.promote(name, "上一个版本号")`。这就是为什么版本永远不覆盖
- 新版本通常先挂 `challenger`，和 `champion` 并排跑一段时间（比如同时给一小部分请求打分），确认没问题再挪 `champion`

### 面试怎么说
> "Each release registers the tested models as new versions tagged with the test run and the gate result, then moves the champion alias. Rolling back is moving the alias to the previous version."

---

## 第 4 步 · 按名字取回

### 在干什么
像 09c 那样，只用名字和标签把模型取回来，并确认取回的就是考过的那个。

### 执行

In [ ]:
head = registry.load(registry.full_name("stability"), alias="champion", flavor="pytorch")
tested_head = registry._mlflow().pytorch.load_model(TESTED["stability"])
torch.manual_seed(0)
embedding, features = torch.randn(5, 1280), torch.randn(5, head.mean.numel())
same = torch.allclose(head(embedding, features), tested_head(embedding, features))
trees = registry.load(registry.full_name("trees"), alias="champion", flavor="xgboost")
tables = registry.distributions(registry.full_name("trees"), alias="champion")
print("取回的稳定性网络和考过的完全一样:", same)
print("树模型:", type(trees).__name__, "| 换算表:", {name: len(values) for name, values in tables.items()})

### 结果怎么看
- `取回的稳定性网络和考过的完全一样: True`：按 `@champion` 取回的网络，和 09a 考试时用的那个，对同样的输入给出同样的输出
- 树模型是 `XGBRegressor`；换算表有三个成员，每个 1,001 个分位点

09c 打分时就是这样取模型：不写任何版本号、任何运行编号。

### 面试怎么说
> "Batch scoring loads models by alias from the registry, together with the percentile tables stored with their training run, and I check that what the alias returns is bit-for-bit the model that was tested."

---

## 提交代码

In [ ]:
run("git add -A")
run("uv run pre-commit run", check=False)
run("git add -A")
run('(git diff --cached --quiet || git commit -q -m "feat: 09b model registry with a release gate") && git push -q')

推送后去 GitHub 的 Actions 页面看一眼，应该全绿。

---

## 本章小结

| 步骤 | 做了什么 | 记住什么 |
|---|---|---|
| 1 注册表 | 名字、版本、标签 | 用的人只认标签 |
| 2 门禁 | 读考试结果，三条全过才往下 | 规则提前定，自动执行 |
| 3 登记 | 新版本带着来历，挂上 champion | 版本不覆盖，回滚是挪标签 |
| 4 取回 | 按名字和标签取模型 | 确认取回的就是考过的 |

**一句话**：考过门禁的两个模型进了注册表、挂上了"冠军"，以后谁要用模型，都只认这个标签。

## 面试题

1. **Why use a model registry instead of a file path?**
   > "Versions are immutable and traceable to their training run and data, permissions are managed with the tables, and consumers load by alias, so releases and rollbacks need no code change."

2. **What is the difference between a version and an alias?**
   > "A version is a fixed, numbered artefact; an alias is a movable pointer to one version, such as champion."

3. **How do you make sure the registered model is the one that was tested?**
   > "The test evaluation records the URIs of the models it evaluated; registration reads them from there, and a check compares the outputs of the alias with the tested model."

**下一步 09c · 全蛋白组打分**：用 `@champion` 的两个模型，给 20,652 个蛋白的每一个可能突变打分。